# 🛠️ Part 2: Prepare & Load Data (独立运行版)
> **注意**: 为了确保这个 Notebook 能独立运行，我们在这里快速重构一下数据。
> (相当于把 Part 1 的核心数据准备工作浓缩到这里)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

# 1. 加载数据
df = pd.read_csv('data/walmart/Walmart_Sales.csv')

# 2. 基础预处理 (和 Part 1 一样)
df.columns = df.columns.str.lower()
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')
df = df.sort_values(by=['store', 'date'])

# 3. 快速构造核心特征 (为了对比模型，用简单的 Lag 特征即可)
df['dayofyear'] = df['date'].dt.dayofyear
df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
for lag in [1, 4, 52]: # 只要这几个核心特征就能跑
    df[f'sales_lag{lag}'] = df.groupby('store')['weekly_sales'].shift(lag)

# 4. 其它特征
df = pd.get_dummies(df, columns=['store'], drop_first=True)
df = df.dropna() # 丢弃开头缺失的行

# 5. 划分 X, y
X = df.drop(['date', 'weekly_sales'], axis=1)
y = df['weekly_sales']

# 6. 切分 Train/Test (按时间切分)
train_size = int(len(df) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print(f"✅ Data Ready: Train={X_train.shape}, Test={X_test.shape}")

✅ Data Ready: Train=(3276, 54), Test=(819, 54)


# ⚔️ 拓展模块 1: Bagging vs Boosting (理论实战)

## 1. 理论核心 (Interview Keypoint)
| 维度 | Bagging (例: Random Forest) | Boosting (例: XGBoost/LGBM) |
| :--- | :--- | :--- |
| **原理** | **并行 (Parallel)**: 训练多棵独立的树，然后投票/平均 | **串行 (Sequential)**: 后一棵树专门修补前一棵树的错误 (残差) |
| **核心作用** | 降低 **方差 (Variance)** (防过拟合 - 稳) | 降低 **偏差 (Bias)** (提升准确度 - 准) |
| **树的类型** | 深树 (强学习器，容易过拟合) | 浅树 (弱学习器，如 depth=3~6) |
| **适用场景** | 数据噪点多，容易过拟合时 | 追求极致准确率，数据较干净时 |

> **🦄 通俗解释**:
> - **Bagging (RF)**: 像一群 **"诸葛亮"** (强模型) 开会。每个人都看一部分数据，可能有偏见，但大家一投票，结果就稳了 (低方差)。
> - **Boosting (XGB)**: 像 **"错题本"** 练习。第一轮做错了，第二轮重点攻克错题。一步步逼近真相 (低偏差)。

In [2]:
# 1. 训练 Random Forest (Bagging 代表)
print("⏳ Training Random Forest (Available cores)... ")
rf_model = RandomForestRegressor(
    n_estimators=100,     # 树的数量
    max_depth=None,       # RF 的树通常很深 (Full grow)
    n_jobs=-1,            # 并行训练 (Bagging的优势！)
    random_state=42
)

rf_model.fit(X_train, y_train)

# 2. 预测 & 评估
rf_pred = rf_model.predict(X_test)
rf_mape = mean_absolute_percentage_error(y_test, rf_pred)

# 3. 训练 XGBoost (Boosting 代表)
print("⏳ Training XGBoost... ")
xgb_model = XGBRegressor(
    n_estimators=100, 
    max_depth=6,          # Boosting 的树通常较浅
    learning_rate=0.1,    # 步长
    n_jobs=-1,
    random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_mape = mean_absolute_percentage_error(y_test, xgb_pred)

print("-" * 30)
print(f"Random Forest MAPE: {rf_mape:.4f}")
print(f"XGBoost MAPE      : {xgb_mape:.4f}")

# 4. 结论
if xgb_mape < rf_mape:
    print("✅ 结论: Boosting 胜出！说明在这个数据集上，我们需要降低 Bias 以获得更高精度。")
else:
    print("✅ 结论: Bagging 胜出！说明数据噪声较大，RF 的稳定性更好。")

⏳ Training Random Forest (Available cores)... 
⏳ Training XGBoost... 
------------------------------
Random Forest MAPE: 0.0478
XGBoost MAPE      : 0.0471
✅ 结论: Boosting 胜出！说明在这个数据集上，我们需要降低 Bias 以获得更高精度。


# 🎛️ 拓展模块 2: 自动调参进化论 (Auto-Tuning Evolution)

## 1. 调参的三重境界
面试官问 "你怎么调参？" 时，可以展示这三种方法的演进：

### **Level 1: Grid Search (网格搜索)** 🕸️
- **原理**: 暴力穷举。你给 `depth=[3,5,7]`, `lr=[0.1, 0.01]`，它就跑 $3 \times 2 = 6$ 次。
- **缺点**: **太慢了！** 随着参数增加，计算量指数级爆炸 (Curse of Dimensionality)。
- **代码**: `sklearn.model_selection.GridSearchCV`

### **Level 2: Random Search (随机搜索)** 🎲
- **原理**: 在参数空间里 **"随机撒网"**。设定跑 50 次，它就随机抽 50 组参数。
- **优点**: **性价比极高**。它不会在没用的参数上浪费时间，有更大概率撞到一个"好参数"。
- **局限**: 它是**"盲目"**的。第 50 次尝试完全不知道第 49 次的结果是好是坏。
- **代码**: `sklearn.model_selection.RandomizedSearchCV` (**面试推荐首选**)

### **Level 3: Bayesian Optimization (贝叶斯优化 - Optuna)** 🧠
- **原理**: **"以史为鉴"**。它建立一个概率模型，根据前 10 次的结果，推断哪里可能是"金矿"，第 11 次就重点挖那里。
- **优点**: **最智能、最高效**。能在更少的尝试次数内找到更好的参数。
- **代码**: `import optuna` (**大厂加分项**)

In [3]:
from sklearn.model_selection import RandomizedSearchCV

# --- 演示 Level 2: RandomizedSearchCV (性价比之王) ---

# 1. 定义参数分布
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9],
}

# 2. 初始化
random_search = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=5,           # ⚠️ 演示只跑 5 次
    scoring='neg_mean_absolute_percentage_error', 
    cv=2,               # 演示用 2 折
    verbose=1,
    n_jobs=-1
)

print("🎲 Starting Randomized Search (Demo 5 iterations)...")
random_search.fit(X_train, y_train)
print(f"Best Parameters: {random_search.best_params_}")

🎲 Starting Randomized Search (Demo 5 iterations)...
Fitting 2 folds for each of 5 candidates, totalling 10 fits
Best Parameters: {'subsample': 0.9, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.05}


In [4]:
# --- 演示 Level 3: Optuna (高阶玩法) ---

try:
    import optuna
    
    def objective(trial):
        # 1. 智能"猜"参数
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'n_jobs': -1,
            'random_state': 42
        }
        
        # 2. 训练
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)
        
        # 3. 评估
        preds = model.predict(X_test)
        return mean_absolute_percentage_error(y_test, preds)

    print("\n🧠 Starting Optuna Optimization...")
    study = optuna.create_study(direction='minimize') 
    study.optimize(objective, n_trials=5) # 演示跑 5 次
    print(f"Best Optuna Params: {study.best_params}")
    
except ImportError:
    print("⚠️ Optuna not installed. Run `pip install optuna` to try Level 3.")

[I 2026-02-15 16:31:31,106] A new study created in memory with name: no-name-9314988b-19bf-47e4-9fbb-fb0b3f151a63



🧠 Starting Optuna Optimization...


[I 2026-02-15 16:31:32,220] Trial 0 finished with value: 0.04771135700235774 and parameters: {'n_estimators': 340, 'max_depth': 10, 'learning_rate': 0.023925273189853692, 'subsample': 0.8398116021113023}. Best is trial 0 with value: 0.04771135700235774.
[I 2026-02-15 16:31:33,128] Trial 1 finished with value: 0.048546202302780256 and parameters: {'n_estimators': 325, 'max_depth': 10, 'learning_rate': 0.049158822465203526, 'subsample': 0.9138099458285561}. Best is trial 0 with value: 0.04771135700235774.
[I 2026-02-15 16:31:34,410] Trial 2 finished with value: 0.046869514472844855 and parameters: {'n_estimators': 491, 'max_depth': 9, 'learning_rate': 0.04440890237155863, 'subsample': 0.8736882891146747}. Best is trial 2 with value: 0.046869514472844855.
[I 2026-02-15 16:31:35,335] Trial 3 finished with value: 0.045247470830933084 and parameters: {'n_estimators': 426, 'max_depth': 9, 'learning_rate': 0.025819204182089075, 'subsample': 0.6131512540629703}. Best is trial 3 with value: 0.04

Best Optuna Params: {'n_estimators': 426, 'max_depth': 9, 'learning_rate': 0.025819204182089075, 'subsample': 0.6131512540629703}
